In [1]:
# 1. 데이터 로드
import numpy as np
import pandas as pd
path = 'C:/Users/User/OneDrive/바탕 화면/'
train_df = pd.read_csv(path + 'train.csv')
test_df = pd.read_csv(path + 'test.csv')
submission = pd.read_csv(path + 'sample_submission.csv')

In [2]:

def preprocess_for_catboost(train_df, test_df):
    train = train_df.copy()
    test = test_df.copy()
    
    # ===============================
    # 1. 타깃 분리
    # ===============================
    y_train = train['credit'].astype(int)
    train = train.drop('credit', axis=1)
    
    full = pd.concat([train, test], axis=0).reset_index(drop=True)
    
    # ===============================
    # 2. user_id 생성 (⭐ 유지!)
    # ===============================
    id_cols = [
        'gender', 'car', 'reality', 'child_num', 'income_total',
        'income_type', 'edu_type', 'family_type', 'house_type',
        'DAYS_BIRTH', 'DAYS_EMPLOYED',
        'work_phone', 'phone', 'email', 'occyp_type'
    ]
    
    full['user_id'] = full[id_cols].astype(str).agg('_'.join, axis=1)

    # ===============================
    # 3. 카드 활동 파생 (강력)
    # ===============================
    full['begin_month'] = -full['begin_month']

    full['user_card_cnt'] = full.groupby('user_id')['user_id'].transform('count')
    full['user_begin_min'] = full.groupby('user_id')['begin_month'].transform('min')
    full['user_begin_max'] = full.groupby('user_id')['begin_month'].transform('max')

    full['user_begin_range'] = full['user_begin_max'] - full['user_begin_min']
    full['user_begin_mean'] = full.groupby('user_id')['begin_month'].transform('mean')
    full['user_begin_std'] = (
        full.groupby('user_id')['begin_month'].transform('std').fillna(0)
    )

    full['months_active'] = full['user_card_cnt']
    full['card_age'] = full['user_begin_max'] - full['begin_month']
    full['is_new_user'] = (full['months_active'] < 3).astype(int)

    # ===============================
    # 4. 수치형 파생 (그대로 유지)
    # ===============================
    full['age'] = -full['DAYS_BIRTH'] / 365.25
    full['employment_years'] = full['DAYS_EMPLOYED'].apply(
        lambda x: -x / 365.25 if x < 0 else 0
    )

    full['income_per_person'] = full['income_total'] / (full['family_size'] + 1)
    full['log_income'] = np.log1p(full['income_total'])
    full['log_income_per_person'] = np.log1p(full['income_per_person'])

    full['age_emp_ratio'] = full['age'] / (full['employment_years'] + 1)
    full['income_begin_ratio'] = full['income_total'] / (full['begin_month'] + 1)

    # ===============================
    # 5. 범주형 컬럼 정리 (❗ 인코딩 안 함)
    # ===============================
    # CatBoost는 NaN 허용 + 문자열 그대로 OK
    cat_cols = [
        'gender', 'car', 'reality',
        'income_type', 'edu_type', 'family_type',
        'house_type', 'occyp_type',
        'user_id'
    ]

    # object가 아닌데 범주 의미인 건 문자열로 캐스팅
    for col in cat_cols:
        full[col] = full[col].astype(str)

    # ===============================
    # 6. 삭제 (CatBoost에 불필요한 컬럼만)
    # ===============================
    drop_cols = [
        'index',
        'DAYS_BIRTH',
        'DAYS_EMPLOYED',
        'FLAG_MOBIL'
    ]
    full = full.drop(columns=[c for c in drop_cols if c in full.columns])

    # ===============================
    # 7. 분리
    # ===============================
    X_train = full.iloc[:len(train)].copy()
    X_test  = full.iloc[len(train):].copy()

    return X_train, y_train, X_test, cat_cols

In [ ]:
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# 전처리 함수 실행 (이 부분이 먼저 정의되어 있어야 합니다)
X_train_cb, y_train_cb, X_test_cb, cat_cols = preprocess_for_catboost(train_df, test_df)

# 1. 설정값 준비
n_splits = 5
n_class = 3
seeds = [0, 1, 2, 42, 77, 202, 777]

# ==========================================
# [STEP 1] Optuna 하이퍼파라미터 튜닝
# ==========================================
def objective(trial):
    param = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 1000, 
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False
    }
    
    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []
    
    # trial 내부에서 데이터를 Pool로 감싸는 것이 좋습니다.
    for tr_idx, va_idx in skf_tune.split(X_train_cb, y_train_cb):
        X_tr, X_va = X_train_cb.iloc[tr_idx], X_train_cb.iloc[va_idx]
        y_tr, y_va = y_train_cb.iloc[tr_idx], y_train_cb.iloc[va_idx]
        
        # [수정] cat_features를 여기서 지정해줍니다.
        train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
        valid_pool = Pool(X_va, y_va, cat_features=cat_cols)

        model = CatBoostClassifier(**param)
        model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=50, use_best_model=True)
        
        preds = model.predict_proba(valid_pool)
        cv_scores.append(log_loss(y_va, preds))
    
    return np.mean(cv_scores)

print("🚀 [STEP 1] Optuna 시작...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20) 
best_params = study.best_params
print("✅ Best Params:", best_params)

# ==========================================
# [STEP 2] 7-Seed Ensemble 루프
# ==========================================
oof_final = np.zeros((len(X_train_cb), n_class))
test_final = np.zeros((len(X_test_cb), n_class))

for seed in seeds:
    print(f"\n==================== Seed {seed} ====================")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_seed = np.zeros((len(X_train_cb), n_class))
    test_seed = np.zeros((len(X_test_cb), n_class))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_cb, y_train_cb), 1):
        X_tr, y_tr = X_train_cb.iloc[tr_idx], y_train_cb.iloc[tr_idx]
        X_va, y_va = X_train_cb.iloc[va_idx], y_train_cb.iloc[va_idx]

        train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
        valid_pool = Pool(X_va, y_va, cat_features=cat_cols)
        test_pool  = Pool(X_test_cb, cat_features=cat_cols)

        model = CatBoostClassifier(
            loss_function="MultiClass",
            eval_metric="MultiClass",
            iterations=5000,
            random_seed=seed,
            verbose=200,
            **best_params 
        )

        model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=100, use_best_model=True)

        oof_seed[va_idx] = model.predict_proba(valid_pool)
        test_seed += model.predict_proba(test_pool) / n_splits

    oof_final += oof_seed / len(seeds)
    test_final += test_seed / len(seeds)

# ==========================================
# [STEP 3] 제출 저장 (경로 수정 필수!)
# ==========================================
cat_submission = submission.copy()
cat_submission["0"] = test_final[:, 0]
cat_submission["1"] = test_final[:, 1]
cat_submission["2"] = test_final[:, 2]

In [ ]:
# [수정] 경로 끝에 파일 이름(.csv)을 꼭 붙여주세요!
save_path_cat = r'C:/Users/User/OneDrive/바탕 화면/submission_catboost_final.csv'
cat_submission.to_csv(save_path_cat, index=False)

print(f"✅ 제출 파일 저장 완료: {save_path_cat}")